# ViEdge-Gov — chạy P0 trên Kaggle T4 x2

**Trước khi chạy:** Settings → Accelerator → **GPU T4 x2**, Internet → **On**.

Notebook này chạy P0: 2 model x 4 mức nén x VMLU-sampled.
Ước tính 3-5 giờ. Kaggle giới hạn 12h/phiên và ~30h/tuần — chia làm 2 phiên,
mỗi phiên 1 model, lưu output ra Dataset để không mất.


In [ ]:
!nvidia-smi
!free -g | head -2

## 1. Lấy repo

In [ ]:
# Cách A: upload viedge-gov.zip làm Kaggle Dataset rồi giải nén
!cp -r /kaggle/input/viedge-gov/* /kaggle/working/ 2>/dev/null || echo "chưa gắn dataset"

# Cách B: clone từ git sau khi đã push
# !git clone https://github.com/<user>/viedge-gov /kaggle/working/viedge-gov

%cd /kaggle/working/viedge-gov
!ls

## 2. Môi trường

Ghim `lm-eval==0.4.12`. Bản 0.4.9 lỗi `AutoModelForVision2Seq`.
**Không** ghim `transformers==4.53.2` trong cùng env với vllm.

In [ ]:
!pip install -q lm-eval==0.4.12 llmcompressor huggingface_hub datasets pyyaml
!pip list 2>/dev/null | grep -Ei "lm-eval|transformers|torch|llmcompressor"

## 3. Kiểm repo trước khi đốt GPU

`make smoke` phải xanh. Đỏ ở đâu sửa ở đó — đừng chạy tiếp.

In [ ]:
import os
os.environ["PYTHONPATH"] = "/kaggle/working/viedge-gov/src"
!python scripts/99_smoke_all.py
!python -m pytest tests/ -q

## 4. Dữ liệu: VMLU + corpus

In [ ]:
!python scripts/01_download_vmlu.py
!python scripts/02_sample_vmlu.py
!cat results/tables/vmlu_sampling_report.json

## 5. Bộ calibration TIẾNG VIỆT

⚠️ Bắt buộc tiếng Việt. Calib bằng C4 tiếng Anh sẽ tự tạo ra chính hiện tượng
suy giảm mà đề tài đang đo → hỏng tính hợp lệ nội tại (docs/DECISIONS.md ADR-007).

In [ ]:
import json, random
from pathlib import Path

# Nguồn ưu tiên: văn bản pháp luật đã bóc + văn xuôi tiếng Việt thường
texts = []
ap = Path("data/processed/articles.jsonl")
if ap.exists():
    texts += [json.loads(l)["text"] for l in ap.read_text(encoding="utf-8").splitlines() if l.strip()]
print("từ corpus pháp luật:", len(texts))

# Bổ sung văn xuôi thường để calib không lệch hoàn toàn về văn phong pháp lý
# (thay bằng nguồn tiếng Việt bất kỳ đã tải sẵn)
# from datasets import load_dataset
# ds = load_dataset("vietgpt/wikipedia_vi", split="train[:500]")
# texts += [r["text"][:2000] for r in ds]

random.seed(20260825); random.shuffle(texts)
out = Path("data/processed/calib_vi.jsonl"); out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8") as f:
    for t in texts[:256]:
        f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
print("calib:", min(len(texts), 256), "mẫu ->", out)

## 6. Xuất các mức nén

Chạy `DRY=1` trước để **xem lệnh** rồi mới đốt GPU.

In [ ]:
!DRY=1 python scripts/04_export_quant.py

In [ ]:
!python scripts/04_export_quant.py
!du -sh models/* 2>/dev/null

## 7. Chạy VMLU → Bảng 3 (RQ1)

In [ ]:
!python scripts/05_run_eval.py
!cat results/tables/degradation_rq1.json

## 8. Sinh tự do → hàng đợi gán nhãn (RQ2)

Đây là đầu vào cho phần **tính mới** của đề tài. Sinh xong tải file về máy
để hai người gán nhãn độc lập bằng `scripts/07_annotate_cli.py`.

In [ ]:
# Sinh đầu ra tự do cho từng biến thể mô hình.
# Định dạng cần: mỗi dòng {"id":..., "prompt":..., "output":...}
# -> results/taxonomy/generations/<model>@<precision>.jsonl
#
# Mở rộng bộ probe lên >=150 prompt trong Tuần 2 (data/processed/probes_vi.jsonl).
!python scripts/06_run_error_probe.py

## 9. Gom bảng và lưu kết quả

Kaggle xoá `/kaggle/working` khi hết phiên. **Luôn lưu ra output.**

In [ ]:
!python scripts/12_make_report_tables.py
!cat results/tables/REPORT_TABLES.md

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/viedge_results", "zip", "results")
print("đã đóng gói results -> /kaggle/working/viedge_results.zip")
print("Nhớ: Save Version -> Save & Run All, rồi tải Output về máy.")